# AeroNet Lite Notebook 1 — Demand Forecasting

This notebook is the ML regression part of the project. It predicts delivery demand using simple explainable features:

- `hour`
- `weekday`
- `temperature`
- `weather_code`
- `zone_density_level`

The default dataset is synthetic so the notebook runs offline. In the final report, explain that this is a proxy for Bike Sharing Demand / delivery demand as allowed by the project brief.

In [ ]:
# Run this once if widgets do not appear:
# pip install ipywidgets

from pathlib import Path
import sys

# Works whether you launch Jupyter from the project root or from notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

from ml_pipeline import make_synthetic_demand_data, train_demand_model, predict_demand

## 1. Load synthetic demand data

The target variable is `demand`. Higher values mean more expected delivery requests in that grid/zone/time situation.

In [ ]:
df = make_synthetic_demand_data(n=700, seed=42)
display(df.head(10))
display(df.describe().round(2))

## 2. Interactive data explorer

Use the dropdown to quickly see how demand changes with different input features.

In [ ]:
def plot_feature(feature):
    plt.figure(figsize=(8, 4.5))
    if feature in ["hour", "weekday", "weather_code", "zone_density_level"]:
        grouped = df.groupby(feature)["demand"].mean()
        plt.bar(grouped.index.astype(str), grouped.values)
        plt.xlabel(feature)
        plt.ylabel("Average demand")
        plt.title(f"Average demand by {feature}")
    else:
        plt.scatter(df[feature], df["demand"], alpha=0.35)
        plt.xlabel(feature)
        plt.ylabel("Demand")
        plt.title(f"Demand vs {feature}")
    plt.grid(True, alpha=0.25)
    plt.show()

interact(
    plot_feature,
    feature=Dropdown(
        options=["hour", "weekday", "temperature", "weather_code", "zone_density_level"],
        value="hour",
        description="Feature",
    ),
);

## 3. Train the regression model

Try both models. Random Forest usually performs better because demand has nonlinear patterns such as rush hours and density effects.

In [ ]:
def train_and_show(model_name):
    result = train_demand_model(model_name=model_name, seed=42)
    display(Markdown(f"### {result.model_name}"))
    print("MAE :", result.mae)
    print("RMSE:", result.rmse)
    display(result.test_frame.head(10))

    plt.figure(figsize=(6, 5))
    plt.scatter(result.test_frame["actual_demand"], result.test_frame["predicted_demand"], alpha=0.5)
    lo = min(result.test_frame["actual_demand"].min(), result.test_frame["predicted_demand"].min())
    hi = max(result.test_frame["actual_demand"].max(), result.test_frame["predicted_demand"].max())
    plt.plot([lo, hi], [lo, hi])
    plt.xlabel("Actual demand")
    plt.ylabel("Predicted demand")
    plt.title("Actual vs Predicted Demand")
    plt.grid(True, alpha=0.25)
    plt.show()

    if hasattr(result.model, "feature_importances_"):
        importance = pd.Series(result.model.feature_importances_, index=result.feature_names).sort_values(ascending=False)
        plt.figure(figsize=(7, 4))
        importance.plot(kind="bar")
        plt.title("Feature importance")
        plt.ylabel("Importance")
        plt.grid(True, axis="y", alpha=0.25)
        plt.show()

interact(
    train_and_show,
    model_name=Dropdown(options=["Linear Regression", "Random Forest"], value="Random Forest", description="Model"),
);

## 4. Interactive demand prediction

Move the sliders to create a new operating condition and estimate demand.

In [ ]:
# Train one final model for manual prediction
final_demand_result = train_demand_model(model_name="Random Forest", seed=42)

def estimate_demand(hour, weekday, temperature, weather_code, zone_density_level):
    pred = predict_demand(
        final_demand_result.model,
        hour=hour,
        weekday=weekday,
        temperature=temperature,
        weather_code=weather_code,
        zone_density_level=zone_density_level,
    )
    print(f"Predicted demand = {pred:.2f} delivery units")
    print("Meaning: this value can be used by the fleet selector to decide how many drones are required.")

interact(
    estimate_demand,
    hour=IntSlider(min=0, max=23, step=1, value=18, description="Hour"),
    weekday=IntSlider(min=0, max=6, step=1, value=2, description="Weekday"),
    temperature=FloatSlider(min=5, max=45, step=0.5, value=30, description="Temp"),
    weather_code=IntSlider(min=0, max=2, step=1, value=0, description="Weather"),
    zone_density_level=IntSlider(min=1, max=3, step=1, value=3, description="Density"),
);

## Viva explanation

- This is a **regression** problem because demand is numeric.
- MAE tells the average absolute error in demand units.
- RMSE penalizes large mistakes more than MAE.
- Random Forest is usually stronger than Linear Regression here because demand changes nonlinearly with rush hour, temperature, and density.